# BoW 기반 텍스트 분석 연습

1. 적절한 데이터셋을 찾거나 생성하고, 적절한 전처리를 진행한다. [01_preprecessing.ipynb]
2. TF-IDF Vectorizer를 이용하여 벡터화한다.
3. Cosine Similarity를 계산하여 입력된 문자열의 긍/부정을 판단한다.
    - [강사님 말씀]
    - 중심점을 잡고 벡터화 해서 유사도 비교...?
    - 이미 긍부정이 있다면 라벨은 그대로 두고 클러스터링 결과를 비교..?

In [6]:
import json

with open("preprocessed_text.json", "r", encoding="utf-8") as f:
    preprocessed_text = json.load(f)

In [7]:
preprocessed_text

[['회사', '문', '닫다', '떠나다', '장사', '안되다', '징징거리다', '상', '인들'],
 ['절절', '맞다', '말씀', '기업', '살다', '나라', '살', '고', '국민', '사다'],
 ['감사하다', '정확하다', '지적', '이십', '니', '다', '삼성', '현대', 'sk', '기업', '우리나라', '응원'],
 ['이재용', '회장', '님', '언제나', '응원', '대한민국', '국민', '국민', '노고', '차다', '치하', '할머니'],
 ['국내', '투자', '기업', '우대', '해주다', '래야', '일자리', '생기', '죠'],
 ['삼성',
  'SK',
  '응원',
  '4',
  '명',
  '먹이다',
  '살리다',
  '초',
  '초소',
  '기업',
  '운영',
  '이리',
  '힘드다',
  '삼성',
  'SK',
  '몇십',
  '만',
  '명',
  '백만',
  '명',
  '먹이다',
  '살리다',
  '두',
  '분',
  '보다',
  '존경',
  '스럽다'],
 ['너무', '좋다', '내용', '기업', '키우다', '국가', '부흥', '응원'],
 ['부산',
  '살다',
  '청주',
  '산지',
  '10년',
  '되어다',
  '전국',
  '다',
  '불경기',
  '인데',
  '청주',
  '하이닉스',
  '때문',
  '인지',
  '호황',
  '기',
  '인거',
  '월세',
  '없다',
  '전국',
  '몰려오다',
  '하이닉스',
  '공',
  '인부',
  '때문',
  '원룸',
  '모텔',
  '아파트',
  '월세',
  '없다',
  '정도',
  '대기업',
  '역활',
  '크다',
  '세',
  '느끼다',
  '한국',
  '계속',
  '경제',
  '발전',
  '모든',
  '국민',
  '노력',
  '해주다',
  '맞다',
  '나라',
  '국민',
  '혜택',
  '

In [ ]:

corpus = [" ".join(tokens) for tokens in preprocessed_text]

# 기준점 설정
standard_texts = [
    "기업 살다 나라 살다 국민 행복하다 응원하다 힘내다", # 긍정 기준 (Index 0)
    "망하다 나라 망조 불경기 힘들다 규제 반대하다"      # 부정 기준 (Index 1)
]

full_corpus = standard_texts + corpus


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF 모델 생성 및 학습
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(full_corpus)

# standard_tfidf: 기준점 벡터 (긍정, 부정 2개)
# content_tfidf: 실제 댓글들 벡터
standard_tfidf = tfidf_matrix[:2]
content_tfidf = tfidf_matrix[2:]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# 긍정 기준점과의 유사도 계산
pos_sim = cosine_similarity(content_tfidf, standard_tfidf[0:1])
# 부정 기준점과의 유사도 계산
neg_sim = cosine_similarity(content_tfidf, standard_tfidf[1:2])

# 결과를 데이터프레임으로 정리
import pandas as pd
result_df = pd.DataFrame({
    'comment': corpus,
    'pos_score': pos_sim.flatten(),
    'neg_score': neg_sim.flatten()
})

# 더 높은 점수를 가진 쪽으로 라벨링
result_df['sentiment'] = result_df.apply(lambda x: 'Positive' if x['pos_score'] > x['neg_score'] else 'Negative', axis=1)

In [12]:
result_df

,comment,pos_score,neg_score,sentiment
0,회사 문 닫다 떠나다 장사 안되다 징징거리다 상 인들,0.000000,0.000000,Negative
1,절절 맞다 말씀 기업 살다 나라 살 고 국민 사다,0.329548,0.049600,Positive
2,감사하다 정확하다 지적 이십 니 다 삼성 현대 sk 기업 우리나라 응원,0.027084,0.000000,Positive
3,이재용 회장 님 언제나 응원 대한민국 국민 국민 노고 차다 치하 할머니,0.084691,0.000000,Positive
4,국내 투자 기업 우대 해주다 래야 일자리 생기 죠,0.027934,0.000000,Positive
...,...,...,...,...
960,광주 보내다 전기 물 땅값 저렴하다 정부 정리 첫 재 표 문 제지 요 아마도 지방 ...,0.000000,0.000000,Negative
961,그렇다 이재명 뽑다 15만원 지 돈 내다 줄 모르다 배급 문학 써다 대면 서 치킨 ...,0.028512,0.000000,Positive
962,경제 망하다 손 비비다 오다 달라 고 애걸 복 걸다,0.000000,0.061106,Negative
963,안타깝다 대기업 그동안 해오다 대한 업보 라 생각 어렵다 힘들다 국민 피빨 먹다 성...,0.034926,0.063659,Negative
